<a href="https://colab.research.google.com/github/NamishBansal15/substation-detection/blob/main/inference-mapping/count-generation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# =========================
# STEP 0 — INSTALLS
# =========================
!pip install geopandas shapely fiona pyproj rtree


# =========================
# STEP 1 — IMPORTS
# =========================
import pandas as pd
import geopandas as gpd
from shapely.geometry import Point
import os
import zipfile


# =========================
# STEP 2 — PATHS
# =========================
BASE_DIR = "/content/drive/MyDrive/dataset-inference/"

PRED_PATH = BASE_DIR + "data/component_predictions.csv"
META_PATH = BASE_DIR + "data/image_metadata.csv"

OUTPUT_STATE = BASE_DIR + "results/state_component_counts.csv"
OUTPUT_NERC  = BASE_DIR + "results/nerc_region_component_counts.csv"

SHAPE_DIR   = "/content/drive/MyDrive/shapefiles/"
ZIP_PATH    = SHAPE_DIR + "states.zip"
STATES_PATH = SHAPE_DIR + "cb_2018_us_state_500k.shp"

# ✅ NEW: NERC GEOJSON
NERC_PATH = BASE_DIR + "data/nerc_gdf.geojson"

os.makedirs(BASE_DIR + "results/", exist_ok=True)


# =========================
# STEP 3 — LOAD DATA
# =========================
preds = pd.read_csv(PRED_PATH)
meta  = pd.read_csv(META_PATH)

preds = preds.rename(columns={"image": "image_path"})


# =========================
# STEP 5 — MERGE (ROBUST)
# =========================
df = preds.merge(meta, on="id", how="left")

print("Merged shape:", df.shape)


# =========================
# STEP 6 — CLEAN COORDS
# =========================
df["latitude"]  = pd.to_numeric(df["latitude"], errors="coerce")
df["longitude"] = pd.to_numeric(df["longitude"], errors="coerce")

df = df.dropna(subset=["latitude", "longitude"])

print("Valid rows:", len(df))


# =========================
# STEP 7 — GEO DATAFRAME
# =========================
gdf = gpd.GeoDataFrame(
    df,
    geometry=gpd.points_from_xy(df["longitude"], df["latitude"]),
    crs="EPSG:4326"
)


# =========================
# STEP 8 — STATES SHAPEFILE
# =========================
if not os.path.exists(STATES_PATH):
    print("Extracting shapefile...")
    with zipfile.ZipFile(ZIP_PATH, 'r') as zip_ref:
        zip_ref.extractall(SHAPE_DIR)

states = gpd.read_file(STATES_PATH).to_crs("EPSG:4326")


# =========================
# STEP 9 — STATE JOIN
# =========================
gdf = gpd.sjoin(gdf, states, how="left", predicate="within")

if "STUSPS" in gdf.columns:
    gdf["STATE"] = gdf["STUSPS"]
elif "NAME" in gdf.columns:
    gdf["STATE"] = gdf["NAME"]
else:
    raise ValueError("State column not found")


# =========================
# STEP 10 — NERC JOIN (NEW ✅)
# =========================
if os.path.exists(NERC_PATH):
    print("Using NERC GeoJSON (better than mapping)...")

    gdf_nerc = gpd.read_file(NERC_PATH).to_crs("EPSG:4326")

    # detect region column
    possible_cols = ["NERC", "region", "NAME"]
    region_col = next((c for c in possible_cols if c in gdf_nerc.columns), None)

    if region_col is None:
        raise ValueError(f"NERC column not found: {gdf_nerc.columns}")

    gdf = gpd.sjoin(gdf, gdf_nerc[[region_col, "geometry"]], how="left", predicate="intersects")

    gdf["NERC"] = gdf[region_col]

else:
    print("Fallback to state → NERC mapping")

    state_to_nerc = {
        "TX": "ERCOT",
        "CA": "WECC","WA": "WECC","OR": "WECC","NV": "WECC",
        "ID": "WECC","UT": "WECC","AZ": "WECC","MT": "WECC",
        "WY": "WECC","CO": "WECC","NM": "WECC",
        "MN": "MRO","ND": "MRO","SD": "MRO","IA": "MRO","WI": "MRO",
        "KS": "SPP","OK": "SPP","NE": "SPP",
        "IL": "RFC","IN": "RFC","OH": "RFC","MI": "RFC",
        "PA": "RFC","NJ": "RFC","MD": "RFC","DE": "RFC",
        "WV": "RFC","VA": "RFC",
        "AL": "SERC","GA": "SERC","FL": "SERC",
        "MS": "SERC","NC": "SERC","SC": "SERC",
        "TN": "SERC","KY": "SERC",
        "NY": "NPCC","VT": "NPCC","NH": "NPCC",
        "ME": "NPCC","MA": "NPCC","CT": "NPCC","RI": "NPCC",
    }

    gdf["NERC"] = gdf["STATE"].map(state_to_nerc)


# =========================
# STEP 11 — CLEAN
# =========================
gdf = gdf.dropna(subset=["STATE", "NERC"])
print("Final rows:", len(gdf))


# =========================
# STEP 12 — SMART AGGREGATION ✅
# =========================
component_cols = [
    "Alt Energy",
    "Circuit Breaker",
    "Reactor",
    "Transformer",
]

print("Aggregating columns:", component_cols)

state_totals = gdf.groupby("STATE")[component_cols].sum()
nerc_totals  = gdf.groupby("NERC")[component_cols].sum()


# =========================
# STEP 13 — SAVE
# =========================
state_totals.to_csv(OUTPUT_STATE)
nerc_totals.to_csv(OUTPUT_NERC)

print("✅ Done!")
print("State totals:", OUTPUT_STATE)
print("NERC totals:", OUTPUT_NERC)